In [ ]:
from pathlib import Path

import datetime
from tqdm import tqdm
import numpy as np
import xarray as xr
import polars as pl
import polars_ds as pds
import polars.selectors as cs
import scoringrules as sr
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, BoundaryNorm
from matplotlib.dates import DateFormatter
from matplotlib.ticker import MaxNLocator
import colormaps
from cartopy import crs as ccrs

from jetutils.definitions import (
    RADIUS,
    DATADIR,
    KAPPA,
    N_WORKERS,
    FIGURES,
    xarray_to_polars,
    do_rle_fill_hole,
    get_index_columns,
    explode_rle,
    do_rle,
    polars_to_xarray,
)
from jetutils.plots import COLORS, make_transparent
from jetutils.data import standardize, flatten_by, compute_anomalies_pl
from jetutils.geospatial import (
    detect_contours,
    jet_integral_haversine,
    haversine,
    inner_detect_contours,
)
from jetutils.jet_finding import (
    find_all_jets,
    preprocess_ds,
    has_periodic_x,
    compute_alignment,
    compute_jet_props,
    is_polar_gmix,
)
from jetutils.derived_quantities import find_axis

basepath = Path(DATADIR).joinpath("s2s")
figures_path = Path(FIGURES).joinpath("Forecast")
%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
# %config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# Jets in forecast

In [ ]:
basepath_fc = basepath.joinpath("wind/daily")
exp_dir = basepath_fc.joinpath("results/1")
exp_dir.mkdir(parents=True, exist_ok=True)
jets_path = exp_dir.joinpath("all_jets.parquet")
props_path = exp_dir.joinpath("props.parquet")

if jets_path.is_file():
    all_jets = pl.read_parquet(jets_path)
else:
    all_jets = []
    for f in tqdm(list(Path(basepath_fc).glob("*.nc"))):
        huh = standardize(xr.open_dataset(f)).expand_dims(forecast_init=[f.stem])
        huh["theta"] = huh["t"] * (1000 / huh["lev"]) ** KAPPA
        huh = huh.drop_vars("t")
        huh["s"] = np.hypot(huh["u"], huh["v"])
        huh = flatten_by(huh, "s")
        all_jets.append(find_all_jets(huh, n_coarsen=1))
    all_jets = pl.concat(all_jets)
    all_jets.write_parquet(jets_path)

if "is_polar" not in all_jets.columns:
    all_jets = is_polar_gmix(all_jets, ["s", "theta"], "week", n_init=5)
    all_jets.write_parquet(jets_path)
else:
    pass

if not props_path.is_file():
    props = compute_jet_props(all_jets)
    props.write_parquet(props_path)
else:
    props = pl.read_parquet(props_path)

# ERA5 jets versus mslp forecast skills

In [ ]:
min_lat = 36
min_len = 13
hole_size = datetime.timedelta(hours=12)
min_month = 5
props_stj = (
    pl.read_parquet(basepath.joinpath("exp9/props_as_df.parquet"))
    .filter(pl.col("jet") == "STJ")
    .drop("jet", "is_polar")
)
df = props_stj[["time", "mean_lat"]].with_columns(year=pl.col("time").dt.year())
df = df.with_columns(
    pl.col("mean_lat")
    .fill_null(strategy="backward")
    .fill_null(strategy="mean")
    .rolling_mean_by("time", window_size="3d")
)
rle = do_rle_fill_hole(
    df,
    (pl.col("mean_lat") > min_lat) & (pl.col("time").dt.month() > min_month),
    ["year"],
    hole_size=hole_size,
)
starts = (
    df.group_by("year", maintain_order=True)
    .agg(len=pl.col("mean_lat").len())
    .with_columns(pl.col("len").cum_sum() - pl.col("len").first())
)
rle = (
    rle.filter(pl.col("value"), pl.col("len") > min_len)
    .group_by("year", maintain_order=True)
    .agg(pl.col("start").first())
    .join(starts, on="year")
    .with_columns(start=pl.col("start") + pl.col("len"))
)
dates = df["time"][rle["start"]].to_frame().with_columns(year=pl.col("time").dt.year())

props_of_interest = ["mean_lat", "mean_theta", "mean_s", "lon_ext", "wavinessDC16"]
props = pl.read_parquet(basepath.joinpath("exp9/props_as_df.parquet"))
props = props["time", "jet", *props_of_interest]
props = compute_anomalies_pl(props, smooth_clim=15, standardize=True)
props = props.pivot("jet", ["STJ", "EDJ"], index="time", separator=":")
props = props.cast({"time": pl.Datetime("ms")})

In [ ]:
ref = standardize(xr.open_dataarray(basepath.joinpath("mslp_era5/daily/full.nc")))
clim = ref.groupby("time.dayofyear").mean()
nyears = np.unique(ref.time.dt.year).shape[0]
clim_unwrapped = ref.copy(data=np.tile(clim.values[:-1], (20, 1, 1)))
clim_std = ref.groupby("time.dayofyear").std()
clim_std_unwrapped = ref.copy(data=np.tile(clim_std.values[:-1], (20, 1, 1)))
crps_clim = sr.crps_normal(ref, clim_unwrapped, clim_std_unwrapped)

In [ ]:
means_spavg = {}
spreads_spavg = {}
means = {}
spreads = {}

ref = standardize(xr.open_dataarray(basepath.joinpath("mslp_era5/daily/full.nc")))
clim = ref.groupby("time.dayofyear").mean()
nyears = np.unique(ref.time.dt.year).shape[0]
clim_unwrapped = ref.copy(data=np.tile(clim.values[:-1], (20, 1, 1)))
clim_std = ref.groupby("time.dayofyear").std()
clim_std_unwrapped = ref.copy(data=np.tile(clim_std.values[:-1], (20, 1, 1)))
crps_clim = ref.copy(
    data=sr.crps_normal(ref, clim_unwrapped, clim_std_unwrapped)
).rename("crps_clim")
crps_clim_df = xarray_to_polars(crps_clim)

for f in np.sort(list(basepath.joinpath("mslp_fct/daily").glob("*.nc"))):
    fct = standardize(xr.open_dataarray(f))
    fct = fct.sel(time=np.isin(fct["time"], ref["time"]))
    ref_ = ref.sel(time=fct["time"])
    score = ref_.copy(
        data=sr.crps_ensemble(ref_, fct.transpose("time", "lat", "lon", "number"))
    ).rename("crps")
    score = (
        xarray_to_polars(score)
        .with_columns(year=pl.col("time").dt.year())
        .join(dates.rename({"time": "jet_shift_time"}), on="year")
        .join(crps_clim_df, on=["time", "lon", "lat"])
        .with_columns(crpss=1 - pl.col("crps") / pl.col("crps_clim"))
    )

    dt = (pl.col("time") - pl.col("time").first().over(pl.col("year"))) / pl.duration(
        days=1
    )
    dtstar = (pl.col("time") - pl.col("jet_shift_time")) / pl.duration(days=1)
    means_spavg[f.stem] = (
        score.group_by("time", maintain_order=True)
        .agg(
            pl.col("crps").mean(),
            pl.col("crpss").mean(),
            pl.col("year").first(),
            pl.col("jet_shift_time").first(),
        )
        .with_columns(dt=dt, dtstar=dtstar)
        # .group_by("dt", maintain_order=True)
        # .agg(pl.col("msl").mean(), pl.col("dtstar"))
    )
    means[f.stem] = score.with_columns(dt=dt, dtstar=dtstar)

    std = (
        xarray_to_polars(fct.std("number").rename("spread"))
        .with_columns(year=pl.col("time").dt.year())
        .join(dates.rename({"time": "jet_shift_time"}), on="year")
    )
    spreads_spavg[f.stem] = (
        std.group_by("time", maintain_order=True)
        .agg(
            pl.col("spread").mean(),
            pl.col("year").first(),
            pl.col("jet_shift_time").first(),
        )
        .with_columns(dt=dt, dtstar=dtstar)
        # .group_by("dt", maintain_order=True)
        # .agg(pl.col("msl").mean(), pl.col("dtstar"))
    )
    spreads[f.stem] = std.with_columns(dt=dt, dtstar=dtstar)
means_df = pl.concat(
    [df_.with_columns(start=pl.lit(name)) for name, df_ in means.items()]
)
spreads_df = pl.concat(
    [df_.with_columns(start=pl.lit(name)) for name, df_ in spreads.items()]
)
means_df = means_df.with_columns(spread=spreads_df["spread"])
means_df = means_df.join(props, on="time")

means_spavg_df = pl.concat(
    [df_.with_columns(start=pl.lit(name)) for name, df_ in means_spavg.items()]
)
spreads_spavg_df = pl.concat(
    [df_.with_columns(start=pl.lit(name)) for name, df_ in spreads_spavg.items()]
)
means_spavg_df = means_spavg_df.with_columns(spread=spreads_spavg_df["spread"])
means_spavg_df = means_spavg_df.join(props, on="time")

means_south_df = (
    means_df.filter(pl.col("lat") < 40)
    .group_by("start", "time", maintain_order=True)
    .agg(cs.numeric().mean(), pl.col("jet_shift_time").first())
)
means_southwest_df = (
    means_df.filter(pl.col("lat") < 40, pl.col("lon") < 10)
    .group_by("start", "time", maintain_order=True)
    .agg(cs.numeric().mean(), pl.col("jet_shift_time").first())
)

## maps

In [ ]:
import polars_statistics as ps

n_ahead = 6
a = "crpss"
b = props_of_interest[0]
jet = "STJ"
expr = ps.pearson(
    pl.col(f"{b}_{jet}").filter(pl.col("dt") == 0).cast(pl.Float64()),
    pl.col(a).filter(pl.col("dt") == n_ahead).cast(pl.Float64()),
)
all_exprs = {
    f"r:{a}:{b}:{jet}": ps.pearson(
        pl.col(f"{b}:{jet}").filter(pl.col("dt") == 0).cast(pl.Float64()),
        pl.col(a).filter(pl.col("dt") == n_ahead).cast(pl.Float64()),
    ).struct.field("estimate")
    for a in ["crpss", "spread"]
    for b in props_of_interest
    for jet in ["STJ", "EDJ"]
} | {
    f"p:{a}:{b}:{jet}": ps.pearson(
        pl.col(f"{b}:{jet}").filter(pl.col("dt") == 0).cast(pl.Float64()),
        pl.col(a).filter(pl.col("dt") == n_ahead).cast(pl.Float64()),
    ).struct.field("p_value")
    for a in ["crpss", "spread"]
    for b in props_of_interest
    for jet in ["STJ", "EDJ"]
}
all_corrs = (
    means_df.filter(pl.col("dt").max().over("start", "lon", "lat") >= n_ahead)
    .group_by("lon", "lat")
    .agg(**all_exprs)
)
all_corrs = polars_to_xarray(all_corrs, index_columns=["lat", "lon"])

In [ ]:
import polars_ds as pds

n_ahead = 6
all_exprs = {
    f"b:{a}:{b}:{jet}": pds.lin_reg_report(
        pl.col(f"{b}:{jet}").filter(pl.col("dt") == 0),
        target=pl.col(a).filter(pl.col("dt") == n_ahead),
        add_bias=True,
        null_policy="skip",
    )
    .struct.field("beta")
    .first()
    for a in ["crpss", "spread"]
    for b in props_of_interest
    for jet in ["STJ", "EDJ"]
} | {
    f"p:{a}:{b}:{jet}": pds.lin_reg_report(
        pl.col(f"{b}:{jet}").filter(pl.col("dt") == 0),
        target=pl.col(a).filter(pl.col("dt") == n_ahead),
        add_bias=True,
        null_policy="skip",
    )
    .struct.field("p>|t|")
    .first()
    for a in ["crpss", "spread"]
    for b in props_of_interest
    for jet in ["STJ", "EDJ"]
}
all_linregs = (
    means_df.filter(pl.col("dt").max().over("start", "lon", "lat") >= n_ahead)
    .group_by("lon", "lat")
    .agg(**all_exprs)
)
all_linregs = polars_to_xarray(all_linregs, index_columns=["lat", "lon"])

In [ ]:
from jetutils.definitions import PRETTIER_VARNAME

Figure = plt.figure(figsize=(10, 11), constrained_layout=True)
figure, huh = Figure.subfigures(2, 1, wspace=0.02, height_ratios=[1, 0.1])
targets = ["crpss", "spread"]
jets = ["STJ", "EDJ"]
cmap = colormaps.BlWhRe
levels = {
    "crpss": MaxNLocator(9, symmetric=True).tick_values(-0.15, 0.15),
    "spread": MaxNLocator(9, symmetric=True).tick_values(-0.15, 0.15),
}
levels = {key: val[np.abs(val) > 1e-7] for key, val in levels.items()}
norms = {key: BoundaryNorm(val, cmap.N, extend="both") for key, val in levels.items()}
lon, lat = all_corrs.lon.values, all_corrs.lat.values
subfigs_targets = figure.subfigures(1, len(targets), wspace=0.02)
i = 0
for target, subfig in zip(targets, subfigs_targets):
    subfigs_jets = subfig.subfigures(1, len(jets), wspace=0.02)
    norm = norms[target]
    im = ScalarMappable(norm=norm, cmap=cmap)
    for jet, fig in zip(jets, subfigs_jets):
        axs = fig.subplots(
            len(props_of_interest), 1, subplot_kw={"projection": ccrs.PlateCarree()}
        )
        for ax, var in zip(axs, props_of_interest):
            full_var = f"r:{target}:{var}:{jet}"
            ax.pcolormesh(lon, lat, all_corrs[full_var], norm=norm, cmap=cmap)
            ax.coastlines()
            pvals = f"p:{target}:{var}:{jet}"
            pvals = all_linregs[pvals]
            pvals_ = pvals.values.ravel()
            sort_order = np.argsort(pvals_)
            try:
                cutoff = np.amax(
                    np.where(
                        pvals_[sort_order] <= np.linspace(0, 1, len(sort_order)) * 0.05
                    )[0]
                )
            except ValueError:
                cutoff = 0
            filter_ = np.argsort(sort_order).reshape(pvals.shape) <= cutoff
            crrr = ax.pcolor(
                lon,
                lat,
                pvals.where(filter_),
                hatch="..",
                facecolor="none",
                edgecolor="yellow",
                hatch_linewidth=2.0,
                linewidth=0,
            )
            if i == 0:
                ax.set_yticks([])
                ax.set_ylabel(PRETTIER_VARNAME.get(var, var))
        i += 1
        fig.suptitle(jet)
    subfig.suptitle(f"Target: {target}")
cax = huh.subplots(1, 1)
figure.colorbar(
    im,
    cax=cax,
    shrink=1,
    label=f"Lag-{n_ahead} correlation",
    orientation="horizontal",
    spacing="proportional",
)
Figure.savefig(figures_path.joinpath("maps_correlation.pdf"))

In [ ]:
from jetutils.definitions import PRETTIER_VARNAME

figure = plt.figure(figsize=(10, 11), constrained_layout=True)
targets = ["crpss", "spread"]
jets = ["STJ", "EDJ"]
cmap = colormaps.BlWhRe
levels = {
    "crpss": MaxNLocator(9, symmetric=True).tick_values(-0.04, 0.04),
    "spread": MaxNLocator(9, symmetric=True).tick_values(-10, 10),
}
levels = {key: val[np.abs(val) > 1e-7] for key, val in levels.items()}
norms = {key: BoundaryNorm(val, cmap.N, extend="both") for key, val in levels.items()}
lon, lat = all_linregs.lon.values, all_linregs.lat.values
subfigs_targets = figure.subfigures(1, len(targets), wspace=0.02, width_ratios=[1, 1])
i = 0
for target, subfig_ in zip(targets, subfigs_targets):
    subfig, huh = subfig_.subfigures(2, 1, height_ratios=[1, 0.1])
    subfigs_jets = subfig.subfigures(1, len(jets), wspace=0.02)
    norm = norms[target]
    im = ScalarMappable(norm=norm, cmap=cmap)
    for jet, fig in zip(jets, subfigs_jets):
        axs = fig.subplots(
            len(props_of_interest), 1, subplot_kw={"projection": ccrs.PlateCarree()}
        )
        for ax, var in zip(axs, props_of_interest):
            full_var = f"b:{target}:{var}:{jet}"
            ax.pcolormesh(lon, lat, all_linregs[full_var], norm=norm, cmap=cmap)
            ax.coastlines()
            pvals = f"p:{target}:{var}:{jet}"
            pvals = all_linregs[pvals]
            pvals_ = pvals.values.ravel()
            sort_order = np.argsort(pvals_)
            try:
                cutoff = np.amax(
                    np.where(
                        pvals_[sort_order] <= np.linspace(0, 1, len(sort_order)) * 0.05
                    )[0]
                )
            except ValueError:
                cutoff = 0
            filter_ = np.argsort(sort_order).reshape(pvals.shape) <= cutoff
            crrr = ax.pcolor(
                lon,
                lat,
                pvals.where(filter_),
                hatch="..",
                facecolor="none",
                edgecolor="yellow",
                hatch_linewidth=2.0,
                linewidth=0,
            )
            if i == 0:
                ax.set_yticks([])
                ax.set_ylabel(PRETTIER_VARNAME.get(var, var))
        i += 1
        fig.suptitle(jet)
    subfig.suptitle(f"Target: {target}")
    cax = huh.subplots(1, 1)
    figure.colorbar(
        im,
        cax=cax,
        shrink=1,
        label=f"Lag-{n_ahead} regression",
        orientation="horizontal",
        spacing="proportional",
    )
figure.savefig(figures_path.joinpath("maps_regression.pdf"))

# basic lines plots

## spread vs STJ lat

In [ ]:
# from matplotlib.dates import MonthLocator
# from jetutils.plots import COLORS

# huh = (
#     means_spavg_df.filter(pl.col("dt") == 10)
#     .group_by(pl.col("time"))
#     .agg(
#         pl.col("spread").first(),
#         pl.col("dt").first(),
#         pl.col("jet_shift_time").first(),
#         pl.col("mean_lat:STJ").first(),
#     )
#     .sort("time")
# )

# fig, axes = plt.subplots(4, 5, figsize=(12, 8), constrained_layout=True, sharey=True)
# axes = axes.ravel()
# twin_axes = []
# for ax, (year, subdf) in zip(
#     axes, huh.group_by(pl.col("time").dt.year(), maintain_order=True)
# ):
#     year = year[0]
#     ax.axvline(subdf["jet_shift_time"].unique().first(), color="black", lw=2, ls="--")
#     ax.plot(subdf["time"], subdf["mean_lat:STJ"], color=COLORS[2], lw=2)
#     twin_ax = ax.twinx()
#     twin_ax.plot(subdf["time"], subdf["spread"], lw=2, zorder=40)
#     ax.xaxis.set_major_locator(MonthLocator())
#     ax.xaxis.set_major_formatter(DateFormatter("%b"))
#     ax.yaxis.tick_right()
#     twin_ax.yaxis.tick_left()
#     ax.set_title(year)
#     twin_axes.append(twin_ax)

# for twin_ax in twin_axes[1:]:
#     twin_ax.sharey(twin_axes[0])

# for year, ax, twin_ax in zip(huh["time"].dt.year().unique().sort(), axes, twin_axes):
#     if year >= 2020:
#         ax.set_xticks(ax.get_xticks(), labels=ax.get_xticklabels(), rotation=0)
#     else:
#         ax.set_xticks(ax.get_xticks(), labels=["" for _ in ax.get_xticks()])

#     if year % 5 == 0:
#         ax.set_yticks([30, 40, 50, 60], labels=["" for _ in range(4)])
#         ax.tick_params(
#             axis="y",
#             left=False,
#             top=False,
#             right=True,
#             bottom=True,
#             labelleft=False,
#             labeltop=False,
#             labelright=False,
#             labelbottom=True,
#         )
#         twin_ax.set_yticks([400, 600], labels=["400", "600"], visible=True)
#         twin_ax.tick_params(
#             axis="y",
#             left=True,
#             top=False,
#             right=False,
#             bottom=False,
#             labelleft=True,
#             labeltop=False,
#             labelright=False,
#             labelbottom=False,
#         )
#     elif year % 5 == 4:
#         ax.set_yticks([30, 40, 50, 60], labels=["30", "40", "50", "60"])
#         ax.tick_params(
#             axis="y",
#             left=False,
#             top=False,
#             right=True,
#             bottom=True,
#             labelleft=False,
#             labeltop=False,
#             labelright=True,
#             labelbottom=True,
#         )
#         twin_ax.set_yticks([400, 600], labels=["400", "600"], visible=True)
#         twin_ax.tick_params(
#             axis="y",
#             left=True,
#             top=False,
#             right=False,
#             bottom=False,
#             labelleft=False,
#             labeltop=False,
#             labelright=False,
#             labelbottom=False,
#         )
#     else:
#         ax.set_yticks([30, 40, 50, 60], labels=["30", "40", "50", "60"])
#         ax.tick_params(
#             axis="y",
#             left=False,
#             top=False,
#             right=True,
#             bottom=True,
#             labelleft=False,
#             labeltop=False,
#             labelright=False,
#             labelbottom=True,
#         )
#         twin_ax.set_yticks([400, 600], labels=["400", "600"], visible=True)
#         twin_ax.tick_params(
#             axis="y",
#             left=True,
#             top=False,
#             right=False,
#             bottom=False,
#             labelleft=False,
#             labeltop=False,
#             labelright=False,
#             labelbottom=False,
#         )

## gb

In [ ]:
from string import ascii_lowercase
to_plot = means_spavg_df.group_by("start", "dt").agg(
    pl.col("spread").mean(),
    pl.col("crpss").mean(),
    pl.col("dtstar").mean(),
    dayofyear=pl.col("time").first().dt.ordinal_day(),
    jst_std=pl.col("jet_shift_time").dt.ordinal_day().std(),
    jst_mean=pl.col("jet_shift_time").dt.ordinal_day().mean(),
)
to_plot = to_plot.with_columns(
    spread_rm=pl.col("spread")
    .rolling_mean_by(pl.col("dayofyear").cast(pl.UInt32()), "20i")
    .over("dt"),
    crpss_rm=pl.col("crpss")
    .rolling_mean_by(pl.col("dayofyear").cast(pl.UInt32()), "20i")
    .over("dt"),
)
jst_mean = to_plot["jst_mean"][0]
jst_std = to_plot["jst_std"][0]
to_plot = polars_to_xarray(to_plot, ["start", "dt"])
fig, axes = plt.subplots(
    2, 2, figsize=(10, 8), sharex="col", sharey="row", constrained_layout=True
)
cmap = colormaps.NCV_bright
n_dt = 14
colors = cmap(np.linspace(0, 1, n_dt))

for color, dt in zip(colors, to_plot.dt.values[:n_dt]):
    dt = int(dt)
    tplt = to_plot.sel(dt=dt)
    x1 = tplt["dayofyear"].values
    axes[0, 0].plot(x1, tplt["spread"], color=color, lw=0.5, zorder=2)
    axes[0, 0].plot(x1, tplt["spread_rm"], color=color, lw=3, alpha=0.7, label=dt)
    axes[1, 0].plot(x1, tplt["crpss"], color=color, lw=0.5, zorder=2)
    axes[1, 0].plot(x1, tplt["crpss_rm"], color=color, lw=3, alpha=0.7)

    x2 = tplt["dtstar"].values
    axes[0, 1].plot(x2, tplt["spread"], color=color, lw=0.5, zorder=2)
    axes[0, 1].plot(x2, tplt["spread_rm"], color=color, lw=3, alpha=0.7)
    axes[1, 1].plot(x2, tplt["crpss"], color=color, lw=0.5, zorder=2)
    axes[1, 1].plot(x2, tplt["crpss_rm"], color=color, lw=3, alpha=0.7)

x1 = to_plot.sel(dt=0)["dayofyear"].values
x2 = to_plot.sel(dt=0)["dtstar"].values
distrib_jst = (
    1
    / (np.sqrt(2 * np.pi) * jst_std)
    * np.exp(-0.5 * (x1 - jst_mean) ** 2 / jst_std**2)
)
for ax in axes[:, 0]:
    ylims = ax.get_ylim()
    factor = (ylims[-1] - ylims[0]) / distrib_jst.max() * 0.85
    ax.fill_between(
        x1, ylims[0], distrib_jst * factor + ylims[0], color="black", alpha=0.3
    )
    ax.set_ylim(*ylims)
for ax in axes[:, 1]:
    ax.axvline(0.0, zorder=-1, color="black", lw=2)
axes[0, 0].set_ylabel("Spatially averaged Spread [hPa]")
axes[1, 0].set_ylabel("Spatially averaged Crpss [~]")
axes[1, 0].set_xlabel("Ordinal Day")
axes[1, 1].set_xlabel("Offset to STJ shift")
fig.legend(
    ncol=n_dt,
    borderpad=0.3,
    labelspacing=0.2,
    handlelength=1.5,
    columnspacing=0.6,
    handletextpad=0.4,
)
for ax, letter in zip(axes.ravel(), ascii_lowercase):
    ax.annotate(
        letter,
        (.01, .86),
        xycoords="axes fraction",
        ha="left",
        va="baseline",
        fontweight="demi",
        bbox={
            "boxstyle": "square, pad=0.1",
            "edgecolor": "none",
            "facecolor": "white",
        },
        usetex=False,
        fontsize=20,
    )
fig.savefig(figures_path.joinpath("yavg_all_europe.pdf"))

In [ ]:
from string import ascii_lowercase
to_plot = means_south_df.group_by("start", "dt").agg(
    pl.col("spread").mean(),
    pl.col("crpss").mean(),
    pl.col("dtstar").mean(),
    dayofyear=pl.col("time").first().dt.ordinal_day(),
    jst_std=pl.col("jet_shift_time").dt.ordinal_day().std(),
    jst_mean=pl.col("jet_shift_time").dt.ordinal_day().mean(),
)
to_plot = to_plot.with_columns(
    spread_rm=pl.col("spread")
    .rolling_mean_by(pl.col("dayofyear").cast(pl.UInt32()), "20i")
    .over("dt"),
    crpss_rm=pl.col("crpss")
    .rolling_mean_by(pl.col("dayofyear").cast(pl.UInt32()), "20i")
    .over("dt"),
)
jst_mean = to_plot["jst_mean"][0]
jst_std = to_plot["jst_std"][0]
to_plot = polars_to_xarray(to_plot, ["start", "dt"])
fig, axes = plt.subplots(
    2, 2, figsize=(10, 8), sharex="col", sharey="row", constrained_layout=True
)
cmap = colormaps.NCV_bright
n_dt = 14
colors = cmap(np.linspace(0, 1, n_dt))

for color, dt in zip(colors, to_plot.dt.values[:n_dt]):
    dt = int(dt)
    tplt = to_plot.sel(dt=dt)
    x1 = tplt["dayofyear"].values
    axes[0, 0].plot(x1, tplt["spread"], color=color, lw=0.5, zorder=2)
    axes[0, 0].plot(x1, tplt["spread_rm"], color=color, lw=3, alpha=0.7, label=dt)
    axes[1, 0].plot(x1, tplt["crpss"], color=color, lw=0.5, zorder=2)
    axes[1, 0].plot(x1, tplt["crpss_rm"], color=color, lw=3, alpha=0.7)

    x2 = tplt["dtstar"].values
    axes[0, 1].plot(x2, tplt["spread"], color=color, lw=0.5, zorder=2)
    axes[0, 1].plot(x2, tplt["spread_rm"], color=color, lw=3, alpha=0.7)
    axes[1, 1].plot(x2, tplt["crpss"], color=color, lw=0.5, zorder=2)
    axes[1, 1].plot(x2, tplt["crpss_rm"], color=color, lw=3, alpha=0.7)

x1 = to_plot.sel(dt=0)["dayofyear"].values
x2 = to_plot.sel(dt=0)["dtstar"].values
distrib_jst = (
    1
    / (np.sqrt(2 * np.pi) * jst_std)
    * np.exp(-0.5 * (x1 - jst_mean) ** 2 / jst_std**2)
)
for ax in axes[:, 0]:
    ylims = ax.get_ylim()
    factor = (ylims[-1] - ylims[0]) / distrib_jst.max() * 0.85
    ax.fill_between(
        x1, ylims[0], distrib_jst * factor + ylims[0], color="black", alpha=0.3
    )
    ax.set_ylim(*ylims)
for ax in axes[:, 1]:
    ax.axvline(0.0, zorder=-1, color="black", lw=2)
axes[0, 0].set_ylabel("Spatially averaged Spread [hPa]")
axes[1, 0].set_ylabel("Spatially averaged Crpss [~]")
axes[1, 0].set_xlabel("Ordinal Day")
axes[1, 1].set_xlabel("Offset to STJ shift")
fig.legend(
    ncol=n_dt,
    borderpad=0.3,
    labelspacing=0.2,
    handlelength=1.5,
    columnspacing=0.6,
    handletextpad=0.4,
)
for ax, letter in zip(axes.ravel(), ascii_lowercase):
    ax.annotate(
        letter,
        (.01, .86),
        xycoords="axes fraction",
        ha="left",
        va="baseline",
        fontweight="demi",
        bbox={
            "boxstyle": "square, pad=0.1",
            "edgecolor": "none",
            "facecolor": "white",
        },
        usetex=False,
        fontsize=20,
    )
fig.savefig(figures_path.joinpath("yavg_south_europe.pdf"))

In [ ]:
from string import ascii_lowercase
to_plot = means_southwest_df.group_by("start", "dt").agg(
    pl.col("spread").mean(),
    pl.col("crpss").mean(),
    pl.col("dtstar").mean(),
    dayofyear=pl.col("time").first().dt.ordinal_day(),
    jst_std=pl.col("jet_shift_time").dt.ordinal_day().std(),
    jst_mean=pl.col("jet_shift_time").dt.ordinal_day().mean(),
)
to_plot = to_plot.with_columns(
    spread_rm=pl.col("spread")
    .rolling_mean_by(pl.col("dayofyear").cast(pl.UInt32()), "20i")
    .over("dt"),
    crpss_rm=pl.col("crpss")
    .rolling_mean_by(pl.col("dayofyear").cast(pl.UInt32()), "20i")
    .over("dt"),
)
jst_mean = to_plot["jst_mean"][0]
jst_std = to_plot["jst_std"][0]
to_plot = polars_to_xarray(to_plot, ["start", "dt"])
fig, axes = plt.subplots(
    2, 2, figsize=(10, 8), sharex="col", sharey="row", constrained_layout=True
)
cmap = colormaps.NCV_bright
n_dt = 14
colors = cmap(np.linspace(0, 1, n_dt))

for color, dt in zip(colors, to_plot.dt.values[:n_dt]):
    dt = int(dt)
    tplt = to_plot.sel(dt=dt)
    x1 = tplt["dayofyear"].values
    axes[0, 0].plot(x1, tplt["spread"], color=color, lw=0.5, zorder=2)
    axes[0, 0].plot(x1, tplt["spread_rm"], color=color, lw=3, alpha=0.7, label=dt)
    axes[1, 0].plot(x1, tplt["crpss"], color=color, lw=0.5, zorder=2)
    axes[1, 0].plot(x1, tplt["crpss_rm"], color=color, lw=3, alpha=0.7)

    x2 = tplt["dtstar"].values
    axes[0, 1].plot(x2, tplt["spread"], color=color, lw=0.5, zorder=2)
    axes[0, 1].plot(x2, tplt["spread_rm"], color=color, lw=3, alpha=0.7)
    axes[1, 1].plot(x2, tplt["crpss"], color=color, lw=0.5, zorder=2)
    axes[1, 1].plot(x2, tplt["crpss_rm"], color=color, lw=3, alpha=0.7)

x1 = to_plot.sel(dt=0)["dayofyear"].values
x2 = to_plot.sel(dt=0)["dtstar"].values
distrib_jst = (
    1
    / (np.sqrt(2 * np.pi) * jst_std)
    * np.exp(-0.5 * (x1 - jst_mean) ** 2 / jst_std**2)
)
for ax in axes[:, 0]:
    ylims = ax.get_ylim()
    factor = (ylims[-1] - ylims[0]) / distrib_jst.max() * 0.85
    ax.fill_between(
        x1, ylims[0], distrib_jst * factor + ylims[0], color="black", alpha=0.3
    )
    ax.set_ylim(*ylims)
for ax in axes[:, 1]:
    ax.axvline(0.0, zorder=-1, color="black", lw=2)
axes[0, 0].set_ylabel("Spatially averaged Spread [hPa]")
axes[1, 0].set_ylabel("Spatially averaged Crpss [~]")
axes[1, 0].set_xlabel("Ordinal Day")
axes[1, 1].set_xlabel("Offset to STJ shift")
fig.legend(
    ncol=n_dt,
    borderpad=0.3,
    labelspacing=0.2,
    handlelength=1.5,
    columnspacing=0.6,
    handletextpad=0.4,
)
for ax, letter in zip(axes.ravel(), ascii_lowercase):
    ax.annotate(
        letter,
        (.01, .86),
        xycoords="axes fraction",
        ha="left",
        va="baseline",
        fontweight="demi",
        bbox={
            "boxstyle": "square, pad=0.1",
            "edgecolor": "none",
            "facecolor": "white",
        },
        usetex=False,
        fontsize=20,
    )
fig.savefig(figures_path.joinpath("yavg_sw_europe.pdf"))

## every year every forecast spread

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(12, 7), constrained_layout=True, sharey=True)
axes = axes.ravel()
cmap = colormaps.NCV_bright
ouais = pl.col("start").str.split("-").list.first().cast(int) < 8
colors = cmap(
    np.linspace(
        0, 1, means_spavg_df.filter(pl.col("dt") == 4, ouais)["start"].unique().shape[0]
    )
)
for ax, year in zip(axes, means_spavg_df["time"].dt.year().unique()):
    hohoho = means_spavg_df.filter(ouais, pl.col("time").dt.year() == year)
    ax.axvline(
        hohoho["jet_shift_time"].unique().first(),
        color="black",
        lw=2,
        ls="dashed",
        zorder=3,
    )
    huhuhu = hohoho.filter(pl.col("dt") == 4)
    for c, (huh, subdf) in zip(colors, hohoho.group_by("start", maintain_order=True)):
        ax.plot(subdf["time"], subdf["spread"], alpha=0.8, lw=1, c=c)
    ax.plot(huhuhu["time"], huhuhu["spread"], color="black", lw=3)
    ax.xaxis.set_major_formatter(DateFormatter("%m-%d"))
    if year >= 2020:
        ax.set_xticks(ax.get_xticks(), labels=ax.get_xticklabels(), rotation=90)
    else:
        ax.set_xticks(ax.get_xticks(), labels=["" for _ in ax.get_xticks()])
    ax.set_title(year)
fig.supylabel("Spatially averaged spread [hPa]")
fig.savefig(figures_path.joinpath("all_spreads.pdf"))

## every year every forecast crpss

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(12, 7), constrained_layout=True, sharey=True)
axes = axes.ravel()
cmap = colormaps.NCV_bright
ouais = pl.col("start").str.split("-").list.first().cast(int) < 8
colors = cmap(
    np.linspace(
        0, 1, means_spavg_df.filter(pl.col("dt") == 4, ouais)["start"].unique().shape[0]
    )
)
for ax, year in zip(axes, means_spavg_df["time"].dt.year().unique()):
    hohoho = means_spavg_df.filter(ouais, pl.col("time").dt.year() == year)
    ax.axvline(
        hohoho["jet_shift_time"].unique().first(),
        color="black",
        lw=2,
        ls="dashed",
        zorder=3,
    )
    huhuhu = hohoho.filter(pl.col("dt") == 4)
    for c, (huh, subdf) in zip(colors, hohoho.group_by("start", maintain_order=True)):
        ax.plot(subdf["time"], subdf["crpss"], alpha=0.8, lw=1, c=c)
    ax.plot(huhuhu["time"], huhuhu["crpss"], color="black", lw=3)
    ax.xaxis.set_major_formatter(DateFormatter("%m-%d"))
    if year >= 2020:
        ax.set_xticks(ax.get_xticks(), labels=ax.get_xticklabels(), rotation=90)
    else:
        ax.set_xticks(ax.get_xticks(), labels=["" for _ in ax.get_xticks()])
    ax.set_title(year)
    ax.set_ylim(-0.2, 1.1)
fig.supylabel("Spatially averaged crpss [~]")
fig.savefig(figures_path.joinpath("all_crpss.pdf"))

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(12, 7), constrained_layout=True, sharey=True)
axes = axes.ravel()
cmap = colormaps.NCV_bright
ouais = pl.col("start").str.split("-").list.first().cast(int) < 8
colors = cmap(
    np.linspace(
        0, 1, means_south_df.filter(pl.col("dt") == 4, ouais)["start"].unique().shape[0]
    )
)
for ax, year in zip(axes, means_south_df["time"].dt.year().unique()):
    hohoho = means_south_df.filter(ouais, pl.col("time").dt.year() == year)
    ax.axvline(
        hohoho["jet_shift_time"].unique().first(),
        color="black",
        lw=2,
        ls="dashed",
        zorder=3,
    )
    huhuhu = hohoho.filter(pl.col("dt") == 4)
    for c, (huh, subdf) in zip(colors, hohoho.group_by("start", maintain_order=True)):
        ax.plot(subdf["time"], subdf["crpss"], alpha=0.8, lw=1, c=c)
    ax.plot(huhuhu["time"], huhuhu["crpss"], color="black", lw=3)
    ax.xaxis.set_major_formatter(DateFormatter("%m-%d"))
    if year >= 2020:
        ax.set_xticks(ax.get_xticks(), labels=ax.get_xticklabels(), rotation=90)
    else:
        ax.set_xticks(ax.get_xticks(), labels=["" for _ in ax.get_xticks()])
    ax.set_title(year)
    ax.set_ylim(-0.2, 1.1)
fig.supylabel("Spatially averaged crpss [~]")
fig.savefig(figures_path.joinpath("se_crpss.pdf"))

# against dtstar

In [ ]:
start_dtstar = dtstar.first().over(["start", pl.col("time").dt.year()])
start_dtstar_cut = (
    start_dtstar.cut(np.arange(-72, 71, 4), labels=np.arange(-74, 73, 4).astype(str))
    .cast(pl.Utf8)  # lol
    .cast(pl.Int32)
)
start_doy = pl.col("time").dt.ordinal_day().first().over("start")
means_spavg_df = means_spavg_df.with_columns(
    start_dtstar=start_dtstar, start_dtstar_cut=start_dtstar_cut, start_doy=start_doy
).sort("start_dtstar", "dt")

In [ ]:
cmap = colormaps.rainbow_dark
unique_cats = means_spavg_df["start_dtstar_cut"].n_unique()
boundaries = np.arange(unique_cats + 5) - 0.5
norm = BoundaryNorm(boundaries, cmap.N)

fig, ax = plt.subplots()
for i, (label, df_) in enumerate(
    means_spavg_df.group_by("start_dtstar_cut", maintain_order=True)
):
    color = cmap(norm(i))
    df_ = df_.group_by("dt", maintain_order=True).agg(pl.col("spread").mean())
    ax.plot(df_["dt"], df_["spread"], color=color, label=label)
ax.legend(
    ncol=4,
    borderpad=0.3,
    labelspacing=0.2,
    loc="lower right",
    handlelength=1.5,
    columnspacing=0.6,
    handletextpad=0.4,
)
ax.set_xlabel("Forecast lead time [d]")
ax.set_ylabel("Spatial average Spread [hPa]")
fig.savefig(figures_path.joinpath("spread_vs_leadtime.pdf"))

In [ ]:
cmap = colormaps.rainbow_dark
unique_cats = means_spavg_df["start_dtstar_cut"].n_unique()
boundaries = np.arange(unique_cats + 5) - 0.5
norm = BoundaryNorm(boundaries, cmap.N)

fig, ax = plt.subplots()
for i, (label, df_) in enumerate(
    means_spavg_df.group_by("start_dtstar_cut", maintain_order=True)
):
    color = cmap(norm(i))
    df_ = df_.group_by("dt", maintain_order=True).agg(pl.col("crpss").mean())
    ax.plot(df_["dt"], df_["crpss"], color=color, label=label)
ax.legend(
    ncol=5,
    borderpad=0.3,
    labelspacing=0.2,
    loc="upper right",
    handlelength=1.5,
    columnspacing=0.6,
    handletextpad=0.4,
)
ax.set_xlabel("Forecast lead time [d]")
ax.set_ylabel("Spatial average crpss [~]")
fig.savefig(figures_path.joinpath("crpss_vs_leadtime.pdf"))

In [ ]:
plt.plot(
    means_spavg_df.group_by("dt", maintain_order=True).agg(
        pds.lin_reg("start_doy", target="spread", add_bias=True).list.get(0)
    )["coeffs"],
    ls="solid",
    color="black",
)
plt.plot(
    means_spavg_df.group_by("dt", maintain_order=True).agg(
        pds.lin_reg_report("start_doy", target="spread", add_bias=True)
        .struct.field("r2")
        .first()
    )["r2"],
    ls="dashed",
    color="black",
)

plt.plot(
    means_spavg_df.group_by("dt", maintain_order=True).agg(
        pds.lin_reg("start_dtstar", target="spread", add_bias=True).list.get(0)
    )["coeffs"],
    ls="solid",
    color=COLORS[2],
)
plt.plot(
    means_spavg_df.group_by("dt", maintain_order=True).agg(
        pds.lin_reg_report("start_dtstar", target="spread", add_bias=True)
        .struct.field("r2")
        .first()
    )["r2"],
    ls="dashed",
    color=COLORS[2],
)

In [ ]:
plt.plot(
    means_spavg_df.group_by("dt", maintain_order=True).agg(
        pds.lin_reg("start_doy", target="crpss", add_bias=True).list.get(0)
    )["coeffs"]
    * 1000,
    ls="solid",
    color="black",
)
plt.plot(
    means_spavg_df.group_by("dt", maintain_order=True).agg(
        pds.lin_reg_report("start_doy", target="crpss", add_bias=True)
        .struct.field("r2")
        .first()
    )["r2"],
    ls="dashed",
    color="black",
)

plt.plot(
    means_spavg_df.group_by("dt", maintain_order=True).agg(
        pds.lin_reg("start_dtstar", target="crpss", add_bias=True).list.get(0)
    )["coeffs"]
    * 1000,
    ls="solid",
    color=COLORS[2],
)
plt.plot(
    means_spavg_df.group_by("dt", maintain_order=True).agg(
        pds.lin_reg_report("start_dtstar", target="crpss", add_bias=True)
        .struct.field("r2")
        .first()
    )["r2"],
    ls="dashed",
    color=COLORS[2],
)

In [ ]:
dt = 10
df_ = means_spavg_df.filter(pl.col("dt") == dt)
plt.scatter(df_["start_dtstar"], df_["spread"], c=df_["start_doy"])

In [ ]:
dt = 10
df_ = means_spavg_df.filter(pl.col("dt") == dt)
plt.scatter(df_["start_dtstar"], df_["crpss"], c=df_["start_doy"])

# jet shift timing

In [ ]:
df = pl.read_parquet(basepath.joinpath("exp9/props_as_df.parquet")).filter(
    pl.col("jet") == "STJ"
)[["time", "mean_lat"]]

fig, axes = plt.subplots(
    4, 5, figsize=(10, 7), sharex="all", sharey="all", constrained_layout=True
)
axes = axes.ravel()

for year, ax in zip(df["time"].dt.year().unique(), axes):
    df_ = df.filter(pl.col("time").dt.year() == year)
    dt = datetime.datetime(year, 1, 1) - datetime.datetime(2025, 1, 1)
    df_ = df_.with_columns(time=pl.col("time") - dt)
    for period, color, lw in zip([3, 12], ["black", COLORS[2]], [1, 3]):
        period = datetime.timedelta(days=period)
        offset = period / 2
        ax.plot(
            df_["time"],
            df_.fill_nan(None)
            .rolling("time", period=period)
            .agg(pl.col("mean_lat").mean())["mean_lat"],
            lw=lw,
            zorder=-lw,
            color=color,
        )
        ax.axhline(min_lat, zorder=-10, ls="dashed", color="black")
    # ax.scatter(dates[year - 2005, "time"] - dt, min_lat, marker="x", color="lime", zorder=50, s=200, lw=3)
    ax.axvline(dates[year - 2005, "time"] - dt, lw=3, color=COLORS[3], zorder=-4)
    ax.xaxis.set_major_formatter(DateFormatter("%m"))
    ax.set_title(year)
fig.supylabel("Jet latitude [°N]")
fig.savefig(figures_path.joinpath("STJ_shift_timing.pdf"))